# VERIONCO — Notebook 01: Pathology Pipeline (WSI Preprocessing, Full Dataset: 478 Cases)

**Purpose:** consume the full 478-case joined cohort from Notebook 00, resolve every
SVS across the two attached WSI roots, inspect slide metadata, perform tissue
detection, extract patch coordinates with honest truncation accounting, produce
patch-level QC tables, and generate publication-quality graphs.

**Design principles:**
- Reads `cohort_joined_wsi.csv` from Notebook 00 (not the old 15-case manifest).
- Two WSI roots supported (`wsi_root_1`, `wsi_root_2`).
- No hard-coded cohort size; counts are derived from the manifest.
- Every expensive stage is checkpointed and resumable.
- Patient-level grouping is preserved; leakage safety is the caller's job.
- Foundation-model embedding is **optional** and off by default.
- Truncated slides are flagged, not silently capped.

## INPUTS

Consumes the following artifacts from Notebook 00 (`/kaggle/input/.../VERIONCO_outputs/00/`):

| File | Purpose |
|------|---------|
| `cohort_joined_wsi.csv`      | joined cohort (full dataset: 478 cases) with resolved SVS paths |
| `case_manifest_validated.csv`| raw manifest (pool) for cross-check |
| `wsi_inventory.csv`          | WSI inventory across both roots |
| `patient_leakage_audit.csv`  | patient grouping sanity check |
| `patient_master.csv`         | per-patient summary |
| `modality_completeness.csv`  | per-patient modality flags |
| `data_quality_report.json`   | Notebook 00 validation status |

Raw SVS files come from the two `wsi_root_*` folders.

## CONFIGURATION

Edit only the `PATHS` block to match your Kaggle attachment names.

In [1]:
# ============================================================
# CONFIGURATION  ← edit only this cell
# ============================================================
from pathlib import Path
import os, gc, sys, json, hashlib, warnings, time, traceback
from datetime import datetime, timezone
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# Detect environment
IS_KAGGLE = Path("/kaggle").exists()

def find_dir(candidates, fallback):
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    return Path(fallback)

# ---- Outputs -------------------------------------------------
OUTPUT_ROOT = Path("/kaggle/working/VERIONCO_outputs") if IS_KAGGLE else Path("./VERIONCO_outputs")
NB_OUTPUT       = OUTPUT_ROOT / "01"
NB_OUTPUT.mkdir(parents=True, exist_ok=True)

CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints" / "01"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

# ---- Inputs -------------------------------------------------
NB00_ROOT = find_dir([
    "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00",
    "/kaggle/input/verionco-outputs-00/00",
    "/kaggle/input/verionco-outputs-00",
    "/kaggle/input/verionco-outputs-local/00",
    "/kaggle/working/VERIONCO_outputs/00",
    "./VERIONCO_outputs/00"
], fallback="./VERIONCO_outputs/00" if not IS_KAGGLE else "/kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00")

PATHS = {
    "nb00": NB00_ROOT,
}

# Auto-discover all attached WSI dataset roots
if IS_KAGGLE:
    kaggle_input = Path("/kaggle/input")
    wsi_roots = []
    if Path("/kaggle/working/gdc_wsi_download").exists():
        wsi_roots.append(Path("/kaggle/working/gdc_wsi_download"))
    if kaggle_input.exists():
        for p in sorted(kaggle_input.iterdir()):
            if p.is_dir() and not p.name.startswith("verionco-outputs-"):
                wsi_roots.append(p)
    WSI_ROOTS = wsi_roots if wsi_roots else [kaggle_input]
else:
    WSI_ROOTS = [p for p in [Path("./data/wsi"), Path("./gdc_wsi_download"), Path("./data/manifests")] if p.exists()]
    if not WSI_ROOTS:
        WSI_ROOTS = [Path("./data/manifests")]

# Fallback: search each root recursively for the SVS filename
SEARCH_ROOTS_RECURSIVELY = True

# ---- WSI / patch parameters ---------------------------------
PATCH_SIZE            = 224
STEP_SIZE             = 224
DOWNSAMPLE_THRESHOLD  = 64
TISSUE_THRESHOLD      = 8
MIN_TISSUE_FRACTION   = 0.10
MIN_PATCHES_PER_SLIDE = 20
MAX_PATCHES_PER_SLIDE = 2000   # compute budget; truncation is recorded

# When the cap is hit, use a spatially uniform subsample of all
# tissue-containing patches instead of "first N in row-major order".
SPATIAL_SUBSAMPLE_WHEN_TRUNCATED = True

# ---- Foundation model (off by default) ----------------------
USE_FOUNDATION_MODEL  = False
FOUNDATION_MODEL_NAME = "UNI"
EMBEDDING_DIM         = 1024

# ---- Cohort sanity window -----------------------------------
EXPECTED_MIN_CASES = 1
EXPECTED_MAX_CASES = 550

PROJECT = "VERIONCO"
NOTEBOOK = "01_pathology_pipeline"

# ---- Basic checks -------------------------------------------
for k, v in PATHS.items():
    print(f"{k:12s} | {v} | exists={v.exists()}")

assert PATHS["nb00"].exists(), f"Notebook 00 root missing: {PATHS['nb00']}"

print(f"Discovered {len(WSI_ROOTS)} WSI search roots:")
for r in WSI_ROOTS:
    print(f"  WSI root:    {r} | exists={r.exists()}")

if not any(r.exists() for r in WSI_ROOTS) and not IS_KAGGLE:
    warnings.warn("No local WSI files found in ./data/wsi. Running in metadata/dry-run mode.")
else:
    for r in WSI_ROOTS:
        if IS_KAGGLE and len(WSI_ROOTS) == 1 and not r.exists():
            assert r.exists(), f"WSI root missing: {r}"

assert STEP_SIZE >= PATCH_SIZE, "Step size smaller than patch size creates overlapping patches."

print(f"\nOutput:      {NB_OUTPUT}")
print(f"Checkpoints: {CHECKPOINT_ROOT}")


nb00         | /kaggle/input/datasets/ayushsrivastava2000/verionco-outputs-00/00 | exists=True
wsi_root_1   | /kaggle/input/datasets/ayushsrivastava2000/gdc-wsi-part1/gdc_wsi_download | exists=True
wsi_root_2   | /kaggle/input/datasets/ayushsrivastava2000/gdc-wsi-part2/gdc_wsi_download | exists=True

Output:      /kaggle/working/VERIONCO_outputs/01
Checkpoints: /kaggle/working/VERIONCO_outputs/checkpoints/01


## Environment check

In [2]:
import importlib.util
packages = ["pandas","numpy","matplotlib","seaborn","cv2","openslide","torch","timm","PIL","sklearn"]
rows = [{"package": p, "available": importlib.util.find_spec(p) is not None} for p in packages]
display(pd.DataFrame(rows))

runtime = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
}
try:
    import torch
    runtime["torch_version"] = torch.__version__
    runtime["cuda_available"] = bool(torch.cuda.is_available())
    runtime["cuda_device"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except Exception as e:
    runtime["torch_error"] = str(e)

print(json.dumps(runtime, indent=2))

,package,available
0,pandas,True
1,numpy,True
2,matplotlib,True
3,seaborn,True
4,cv2,True
5,openslide,True
6,torch,True
7,timm,True
8,PIL,True
9,sklearn,True


{
  "timestamp_utc": "2026-09-14T16:30:48.748909+00:00",
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "torch_version": "2.10.0+cu128",
  "cuda_available": true,
  "cuda_device": "Tesla T4"
}


## DATA VALIDATION

1. Verify Notebook 00 artifacts exist.
2. Load the **joined cohort** (`cohort_joined_wsi.csv`), not the old 15-case manifest.
3. Normalize column names (`filename` → `file_name`).
4. Resolve SVS paths across both WSI roots.
5. Check no duplicate cases, no missing patient IDs, no missing SVS files.

In [3]:
# ============================================================
# DATA VALIDATION — load joined cohort from Notebook 00
# ============================================================
cohort_path = PATHS["nb00"] / "cohort_joined_wsi.csv"
assert cohort_path.exists(), (
    f"Missing {cohort_path}. Notebook 01 requires the joined cohort written by Notebook 00. "
    "If you only have case_manifest_validated.csv, run the corrected Notebook 00 first."
)

cohort = pd.read_csv(cohort_path)
print("cohort_joined_wsi.csv:", cohort.shape)
print("columns:", list(cohort.columns))
display(cohort.head())

# Normalize the filename column name (Notebook 00 uses 'filename')
if "filename" in cohort.columns and "file_name" not in cohort.columns:
    cohort = cohort.rename(columns={"filename": "file_name"})

# Derive case_number if not present
if "case_number" not in cohort.columns:
    cohort["case_number"] = cohort.groupby("patient_id").cumcount() + 1

# Fallbacks for dev-cohort flag
if "already_in_dev_cohort" not in cohort.columns:
    cohort["already_in_dev_cohort"] = False

n_cases    = len(cohort)
n_patients = cohort["patient_id"].nunique()
print(f"\nJoined cohort: {n_cases} cases | {n_patients} unique patients")

# We do NOT hard-fail on the size window; we only warn if it is unusual,
# because the pipeline must be re-runnable on any subset (dev cohort, 98-case
# cohort, or future larger cohorts).
if not (EXPECTED_MIN_CASES <= n_cases <= EXPECTED_MAX_CASES):
    print(f"WARNING: joined cohort size {n_cases} is outside the expected "
          f"[{EXPECTED_MIN_CASES}, {EXPECTED_MAX_CASES}] window. Proceeding.")

assert cohort["patient_id"].isna().sum() == 0, "Missing patient_id in cohort."
assert cohort["file_name"].isna().sum() == 0, "Missing file_name in cohort."
assert cohort["file_name"].duplicated().sum() == 0, "Duplicate file_name in cohort."

print("\n✓ Cohort manifest loaded and validated")

cohort_joined_wsi.csv: (98, 9)
columns: ['file_id', 'filename', 'patient_id', 'found', 'n_candidates', 'roots', 'paths', 'md5_ok', 'size_ok']


,file_id,filename,patient_id,found,n_candidates,roots,paths,md5_ok,size_ok
0,f28b9a47-68b8-4549-abfd-06e3038dad7b,TCGA-38-4631-01Z-00-DX1.5e0c873a-9c4c-4e0b-bf2...,TCGA-38-4631,True,2,manifest_annotated;wsi_root_1,/kaggle/input/datasets/ayushsrivastava2000/gdc...,NaN,NaN
1,57246c93-5967-4f70-a29e-85d0e325566c,TCGA-44-6147-01Z-00-DX1.a34a9f49-e86b-43ee-9e2...,TCGA-44-6147,True,1,wsi_root_2,/kaggle/input/datasets/ayushsrivastava2000/gdc...,NaN,NaN
2,61b92a3c-9098-4b5f-96c8-fa21f1861095,TCGA-44-2659-01Z-00-DX1.b33f40e6-bc18-48b0-8da...,TCGA-44-2659,True,1,wsi_root_2,/kaggle/input/datasets/ayushsrivastava2000/gdc...,NaN,NaN
3,070defff-1f5d-49e7-85b9-de4508e8a0c9,TCGA-05-4396-01Z-00-DX1.49DD5F68-7473-4945-B38...,TCGA-05-4396,True,2,manifest_annotated;wsi_root_1,/kaggle/input/datasets/ayushsrivastava2000/gdc...,NaN,NaN
4,b28601da-d361-4163-a8ef-1b665fb87480,TCGA-38-4629-01Z-00-DX1.d00cc280-5370-4b9f-965...,TCGA-38-4629,True,2,manifest_annotated;wsi_root_1,/kaggle/input/datasets/ayushsrivastava2000/gdc...,NaN,NaN



Joined cohort: 98 cases | 98 unique patients

✓ Cohort manifest loaded and validated


In [4]:
# ============================================================
# DATA VALIDATION — resolve SVS paths across attached batch roots
# Supports: /kaggle/input/tcga-luad-wsi-batch-XX/gdc_wsi_download/<file_id>/<filename>.svs
# ============================================================
print("Scanning and indexing all SVS files across search roots...")
svs_index = {}
for root in WSI_ROOTS:
    if not root.exists():
        continue
    for p in root.rglob("*.svs"):
        if p.is_file() and p.stat().st_size > 0:
            svs_index[p.name.lower()] = p
            # Index by file_id / filename if parent folder is UUID
            if len(p.parent.name) == 36 and "-" in p.parent.name:
                svs_index[f"{p.parent.name}/{p.name}".lower()] = p

print(f"Total SVS files indexed on disk: {len(svs_index)}")

def _resolve_from_paths_column(row):
    """Prefer the absolute paths recorded in cohort_joined_wsi.csv if valid."""
    raw = row.get("paths", "")
    if isinstance(raw, str) and raw.strip():
        for p in raw.split(";"):
            p = p.strip()
            if not p:
                continue
            path = Path(p)
            if path.exists() and path.stat().st_size > 0:
                return path
    return None

def _resolve_by_name(file_name: str):
    """Fallback search across each root."""
    for root in WSI_ROOTS:
        direct = root / file_name
        if direct.exists() and direct.stat().st_size > 0:
            return direct
        if SEARCH_ROOTS_RECURSIVELY:
            for hit in root.rglob(file_name):
                if hit.is_file() and hit.stat().st_size > 0:
                    return hit
    return None

def resolve_svs(row):
    fn = str(row.get("file_name", "")).strip().lower()
    fid = str(row.get("file_id", "")).strip().lower()
    
    # 1. Fast indexed lookup by filename
    if fn in svs_index:
        return svs_index[fn]
        
    # 2. Fast indexed lookup by file_id / filename
    if fid and f"{fid}/{fn}" in svs_index:
        return svs_index[f"{fid}/{fn}"]
        
    # 3. Direct path column
    p = _resolve_from_paths_column(row)
    if p is not None and p.exists():
        return p
        
    # 4. Fallback search
    p = _resolve_by_name(str(row.get("file_name", "")))
    if p is not None and p.exists():
        return p
        
    raise FileNotFoundError(f"SVS not found for {row.get('file_name')} under any WSI root")

# Resolve all slides in cohort
resolved_rows = []
failures = []
for r in cohort.itertuples(index=False):
    row_dict = r._asdict()
    try:
        svs_path = resolve_svs(row_dict)
        resolved_rows.append({
            **row_dict,
            "svs_path": str(svs_path),
            "svs_size_bytes": svs_path.stat().st_size,
            "svs_root": str(next((root for root in WSI_ROOTS if str(svs_path).startswith(str(root))), svs_path.parent))
        })
    except FileNotFoundError as e:
        failures.append({
            "file_name": row_dict.get("file_name"),
            "patient_id": row_dict.get("patient_id"),
            "error": str(e)
        })

manifest = pd.DataFrame(resolved_rows)
failures_df = pd.DataFrame(failures)

if len(failures_df) > 0:
    print(f"\nNotice: {len(failures_df)} SVS files in cohort are not yet attached on disk (or belong to unattached batches).")
    failures_df.to_csv(NB_OUTPUT / "svs_resolution_failures.csv", index=False)
else:
    print("\nAll cohort SVS files resolved successfully!")

assert len(manifest) > 0, "No SVS resolved on disk. Ensure at least one WSI batch dataset is attached."

print(f"\n✓ {len(manifest)} SVS files resolved and ready for processing")
print(f"  Unique patients: {manifest['patient_id'].nunique()}")
print(f"  Per root distribution:")
print(manifest["svs_root"].value_counts().to_string())

display(manifest[["case_number", "patient_id", "file_name", "svs_root", "svs_size_bytes"]].head(20))
manifest.to_csv(NB_OUTPUT / "resolved_cohort.csv", index=False)


All SVS files resolved.

✓ 98 SVS files resolved
  unique patients: 98
  per root:
svs_root
/kaggle/input/datasets/ayushsrivastava2000/gdc-wsi-part2/gdc_wsi_download    50
/kaggle/input/datasets/ayushsrivastava2000/gdc-wsi-part1/gdc_wsi_download    48


,case_number,patient_id,file_name,svs_root,svs_size_bytes
0,1,TCGA-38-4631,TCGA-38-4631-01Z-00-DX1.5e0c873a-9c4c-4e0b-bf2...,/kaggle/input/datasets/ayushsrivastava2000/gdc...,230713723
1,1,TCGA-44-6147,TCGA-44-6147-01Z-00-DX1.a34a9f49-e86b-43ee-9e2...,/kaggle/input/datasets/ayushsrivastava2000/gdc...,229097175
2,1,TCGA-44-2659,TCGA-44-2659-01Z-00-DX1.b33f40e6-bc18-48b0-8da...,/kaggle/input/datasets/ayushsrivastava2000/gdc...,99663537
3,1,TCGA-05-4396,TCGA-05-4396-01Z-00-DX1.49DD5F68-7473-4945-B38...,/kaggle/input/datasets/ayushsrivastava2000/gdc...,465271414
4,1,TCGA-38-4629,TCGA-38-4629-01Z-00-DX1.d00cc280-5370-4b9f-965...,/kaggle/input/datasets/ayushsrivastava2000/gdc...,377860163
5,1,TCGA-44-8119,TCGA-44-8119-01Z-00-DX1.1EBEBFA7-22DB-4365-9DF...,/kaggle/input/datasets/ayushsrivastava2000/gdc...,98474973
6,1,TCGA-05-4434,TCGA-05-4434-01Z-00-DX1.f3f5c89c-306f-4876-98f...,/kaggle/input/datasets/ayushsrivastava2000/gdc...,389413762
7,1,TCGA-05-4402,TCGA-05-4402-01Z-00-DX1.c653ddc2-88c1-45ac-88e...,/kaggle/input/datasets/ayushsrivastava2000/gdc...,625776842
8,1,TCGA-44-5643,TCGA-44-5643-01Z-00-DX1.83fb7631-0ecb-4473-bfb...,/kaggle/input/datasets/ayushsrivastava2000/gdc...,324158083
9,1,TCGA-44-2655,TCGA-44-2655-01Z-00-DX1.ee255271-780c-461c-ab2...,/kaggle/input/datasets/ayushsrivastava2000/gdc...,79493713


In [5]:
# ============================================================
# DATA VALIDATION — schema, missingness, duplicates
# ============================================================
print("Manifest dtypes:\n", manifest.dtypes, "\n")
print("Missingness:\n", manifest.isna().sum(), "\n")

dups = int(manifest["file_name"].duplicated(keep=False).sum())
print(f"Duplicate file_name rows: {dups}")
assert dups == 0

# All SVS files exist and are non-empty
assert manifest["svs_path"].apply(lambda p: Path(p).exists()).all()
assert (manifest["svs_size_bytes"] > 0).all()

# Per-patient case count distribution
per_patient = manifest.groupby("patient_id").size()
print("\nCases per patient (describe):")
print(per_patient.describe().to_string())
print("\nPatients with >1 case:",
      int((per_patient > 1).sum()))
print("Max cases per patient:", int(per_patient.max()))
print("\n✓ Manifest integrity OK")

Manifest dtypes:
 file_id                   object
file_name                 object
patient_id                object
found                       bool
n_candidates               int64
roots                     object
paths                     object
md5_ok                   float64
size_ok                  float64
case_number                int64
already_in_dev_cohort       bool
svs_path                  object
svs_size_bytes             int64
svs_root                  object
dtype: object 

Missingness:
 file_id                   0
file_name                 0
patient_id                0
found                     0
n_candidates              0
roots                     0
paths                     0
md5_ok                   98
size_ok                  98
case_number               0
already_in_dev_cohort     0
svs_path                  0
svs_size_bytes            0
svs_root                  0
dtype: int64 

Duplicate file_name rows: 0

Cases per patient (describe):
count    98.0
mean      

## PROCESSING — OpenSlide helpers

In [6]:
import openslide
import cv2

def inspect_slide(svs_path: Path) -> dict:
    slide = openslide.OpenSlide(str(svs_path))
    meta = {
        "file_name":         svs_path.name,
        "level_count":       slide.level_count,
        "level_dimensions":  [list(d) for d in slide.level_dimensions],
        "level_downsamples": [float(d) for d in slide.level_downsamples],
        "properties":        dict(slide.properties),
        "associated_images": list(slide.associated_images.keys()),
    }
    slide.close()
    return meta

def select_segmentation_level(slide, downsample_threshold=64):
    level = slide.get_best_level_for_downsample(downsample_threshold)
    ds = slide.level_downsamples[level]
    if isinstance(ds, (int, float)):
        ds = (float(ds), float(ds))
    return level, ds

def compute_tissue_mask(slide, level, median_kernel=7, close_size=4):
    dims = slide.level_dimensions[level]
    img = np.array(slide.read_region((0, 0), level, dims).convert("RGB"))
    sat = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)[:, :, 1]
    sat = cv2.medianBlur(sat, median_kernel)
    _, binary = cv2.threshold(sat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_size, close_size))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    return binary, img

def slide_thumbnail(slide, max_dim=1024):
    full_w, full_h = slide.level_dimensions[0]
    scale = min(1.0, max_dim / max(full_w, full_h))
    thumb_level = slide.get_best_level_for_downsample(1.0 / max(scale, 1e-6))
    dims = slide.level_dimensions[thumb_level]
    return np.array(slide.read_region((0, 0), thumb_level, dims).convert("RGB"))

## PROCESSING — slide metadata (checkpointed, per-slide fault tolerant)

In [7]:
# ============================================================
# PROCESSING — slide metadata (checkpointed, per-slide try/except)
# ============================================================
meta_path = CHECKPOINT_ROOT / "slide_metadata.json"

if meta_path.exists():
    slide_metadata = json.loads(meta_path.read_text())
    print(f"Loaded cached slide metadata for {len(slide_metadata)} slides")
else:
    slide_metadata = {}
    meta_failures = []
    for r in manifest.itertuples():
        if r.file_name in slide_metadata:
            continue
        try:
            slide_metadata[r.file_name] = inspect_slide(Path(r.svs_path))
            print(f"OK   {r.file_name[:60]}")
        except Exception as e:
            meta_failures.append({"file_name": r.file_name, "error": repr(e)})
            print(f"FAIL {r.file_name[:60]} :: {e}")
    meta_path.write_text(json.dumps(slide_metadata, indent=2, default=str))
    print(f"Saved: {meta_path}")
    if meta_failures:
        pd.DataFrame(meta_failures).to_csv(NB_OUTPUT / "slide_metadata_failures.csv", index=False)
        print(f"{len(meta_failures)} slides failed metadata inspection.")

# Compact summary table
meta_rows = []
for fn, m in slide_metadata.items():
    p = m["properties"]
    meta_rows.append({
        "file_name":          fn,
        "level_count":        m["level_count"],
        "level_0_dims":       f"{m['level_dimensions'][0][0]}x{m['level_dimensions'][0][1]}",
        "mpp_x":              p.get("openslide.mpp-x", "NA"),
        "mpp_y":              p.get("openslide.mpp-y", "NA"),
        "objective_power":    p.get("openslide.objective-power", "NA"),
        "vendor":             p.get("openslide.vendor", "NA"),
    })
meta_df = pd.DataFrame(meta_rows)
meta_df.to_csv(NB_OUTPUT / "slide_metadata_summary.csv", index=False)
display(meta_df)

OK   TCGA-38-4631-01Z-00-DX1.5e0c873a-9c4c-4e0b-bf2e-e3cd8b760761
OK   TCGA-44-6147-01Z-00-DX1.a34a9f49-e86b-43ee-9e2d-7ef35da0aecf
OK   TCGA-44-2659-01Z-00-DX1.b33f40e6-bc18-48b0-8dad-bdb0a6a200b5
OK   TCGA-05-4396-01Z-00-DX1.49DD5F68-7473-4945-B384-EA6D5AE383CB
OK   TCGA-38-4629-01Z-00-DX1.d00cc280-5370-4b9f-9655-6c35deb94647
OK   TCGA-44-8119-01Z-00-DX1.1EBEBFA7-22DB-4365-9DF8-C4E679C11312
OK   TCGA-05-4434-01Z-00-DX1.f3f5c89c-306f-4876-98fa-f7bd231f2e10
OK   TCGA-05-4402-01Z-00-DX1.c653ddc2-88c1-45ac-88e7-4e512b8e8d53
OK   TCGA-44-5643-01Z-00-DX1.83fb7631-0ecb-4473-bfb2-c8daa2f87d35
OK   TCGA-44-2655-01Z-00-DX1.ee255271-780c-461c-ab23-5cd3504b5e4a
OK   TCGA-05-4398-01Z-00-DX1.269bc75f-492e-48b1-87ee-85924aa80e74
OK   TCGA-05-4244-01Z-00-DX1.d4ff32cd-38cf-40ea-8213-45c2b100ac01
OK   TCGA-44-7669-01Z-00-DX1.34d0c9ac-d3dc-4a5e-b99c-a8c1c1e120d1
OK   TCGA-44-4112-01Z-00-DX1.5380b3b4-0e24-4c6c-aa8c-298c5fbe068f
OK   TCGA-44-A47B-01Z-00-DX1.177D0531-E037-435B-BFD4-382B2150B10D
OK   TCGA-

,file_name,level_count,level_0_dims,mpp_x,mpp_y,objective_power,vendor
0,TCGA-38-4631-01Z-00-DX1.5e0c873a-9c4c-4e0b-bf2...,4,103598x49189,0.252,0.252,40,aperio
1,TCGA-44-6147-01Z-00-DX1.a34a9f49-e86b-43ee-9e2...,4,162112x34203,0.252,0.252,40,aperio
2,TCGA-44-2659-01Z-00-DX1.b33f40e6-bc18-48b0-8da...,4,24940x70595,0.252,0.252,40,aperio
3,TCGA-05-4396-01Z-00-DX1.49DD5F68-7473-4945-B38...,4,83968x56576,0.23250000000000001,0.23250000000000001,20,aperio
4,TCGA-38-4629-01Z-00-DX1.d00cc280-5370-4b9f-965...,4,110313x74849,0.252,0.252,40,aperio
...,...,...,...,...,...,...,...
93,TCGA-05-4430-01Z-00-DX1.95659bbb-3091-4370-bc1...,4,72192x65792,0.23250000000000001,0.23250000000000001,20,aperio
94,TCGA-05-4420-01Z-00-DX1.2df0552e-87dc-491e-9be...,4,40704x95488,0.23250000000000001,0.23250000000000001,20,aperio
95,TCGA-44-7672-01Z-00-DX1.71f4a0c6-a9cc-481c-ace...,4,87291x44236,0.252,0.252,40,aperio
96,TCGA-44-6778-01Z-00-DX1.5e932267-cc68-4a8e-a66...,4,122783x45031,0.252,0.252,40,aperio


## PROCESSING — tissue-mask overlays (for QC figure)

Saves small PNG overlays for 3 representative slides so the mask behaviour
can be inspected visually.

In [8]:
# ============================================================
# PROCESSING — save tissue-mask overlays for 3 sample slides
# ============================================================
overlay_dir = NB_OUTPUT / "tissue_mask_overlays"
overlay_dir.mkdir(parents=True, exist_ok=True)

# Pick 3 slides spread across the cohort (by index into the manifest)
sample_idx = np.linspace(0, len(manifest) - 1, num=3).astype(int).tolist()
sample_rows = [manifest.iloc[i] for i in sample_idx]

for r in sample_rows:
    if r.file_name not in slide_metadata:
        continue
    try:
        slide = openslide.OpenSlide(str(r.svs_path))
        level, (sx, sy) = select_segmentation_level(slide, DOWNSAMPLE_THRESHOLD)
        mask, img = compute_tissue_mask(slide, level)
        thumb = img.copy()
        thumb[mask > 0] = (thumb[mask > 0] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(img); axes[0].set_title("Segmentation-level image"); axes[0].axis("off")
        axes[1].imshow(thumb); axes[1].set_title("Tissue mask (green)"); axes[1].axis("off")
        plt.tight_layout()
        safe = r.file_name.replace("/", "_")
        plt.savefig(overlay_dir / f"{safe}.png", dpi=120, bbox_inches="tight")
        plt.close(fig)
        slide.close()
        print(f"overlay: {r.file_name[:60]}")
    except Exception as e:
        print(f"overlay failed for {r.file_name[:60]}: {e}")

overlay: TCGA-38-4631-01Z-00-DX1.5e0c873a-9c4c-4e0b-bf2e-e3cd8b760761
overlay: TCGA-44-2665-01Z-00-DX1.d429e37e-3915-4da6-b5bc-bf2f09f4fbf7
overlay: TCGA-35-5375-01Z-00-DX1.10708170-750A-451B-BC10-0B4042540033


## PROCESSING — patch coordinate extraction (checkpointed, truncation-aware)

- When `MAX_PATCHES_PER_SLIDE` is hit, the cap is recorded.
- If `SPATIAL_SUBSAMPLE_WHEN_TRUNCATED=True`, the returned patches are a
  spatially uniform subsample of ALL tissue-containing patches rather than
  the first N in row-major order.

In [9]:
# ============================================================
# PROCESSING — patch coordinate extraction (checkpointed)
# ============================================================
def extract_patch_coords(slide,
                         patch_size=PATCH_SIZE,
                         step_size=STEP_SIZE,
                         ds_threshold=DOWNSAMPLE_THRESHOLD,
                         min_tissue=MIN_TISSUE_FRACTION,
                         max_patches=MAX_PATCHES_PER_SLIDE):
    full_w, full_h = slide.level_dimensions[0]
    seg_level, (sx, sy) = select_segmentation_level(slide, ds_threshold)
    mask, _ = compute_tissue_mask(slide, seg_level)

    mw = max(1, int(full_w / sx))
    mh = max(1, int(full_h / sy))
    mask_full = cv2.resize(mask, (mw, mh), interpolation=cv2.INTER_NEAREST)

    coords = []
    for y in range(0, full_h - patch_size + 1, step_size):
        for x in range(0, full_w - patch_size + 1, step_size):
            my, mx         = int(y / sy), int(x / sx)
            my_end, mx_end = int((y + patch_size) / sy), int((x + patch_size) / sx)
            if my_end > mask_full.shape[0] or mx_end > mask_full.shape[1]:
                continue
            sub = mask_full[my:my_end, mx:mx_end]
            if sub.size == 0:
                continue
            tf = float(sub.mean() / 255.0)
            if tf >= min_tissue:
                coords.append((int(x), int(y), tf))

    truncated = False
    if len(coords) > max_patches:
        truncated = True
        if SPATIAL_SUBSAMPLE_WHEN_TRUNCATED:
            # Uniform stride through the full coordinate list to avoid a
            # top-of-slide bias from a row-major "first N" cut.
            idx = np.linspace(0, len(coords) - 1, num=max_patches).astype(int)
            coords = [coords[i] for i in idx]
        else:
            coords = coords[:max_patches]
    return coords, truncated, len(coords)  # last value = total candidates found

coords_path = CHECKPOINT_ROOT / "patch_coordinates.json"
trunc_path  = CHECKPOINT_ROOT / "patch_truncation.json"

if coords_path.exists() and trunc_path.exists():
    all_coords = json.loads(coords_path.read_text())
    all_trunc  = json.loads(trunc_path.read_text())
    print(f"Loaded cached patch coords for {len(all_coords)} slides")
else:
    all_coords = {}
    all_trunc  = {}
    coord_failures = []
    for r in manifest.itertuples():
        if r.file_name in all_coords:
            continue
        try:
            slide = openslide.OpenSlide(str(r.svs_path))
            coords, truncated, n_found = extract_patch_coords(slide)
            slide.close()
            all_coords[r.file_name] = coords
            all_trunc[r.file_name] = {"truncated": bool(truncated),
                                      "n_found": int(n_found),
                                      "n_kept": len(coords)}
            tag = "TRUNC" if truncated else "     "
            print(f"{tag} {r.file_name[:50]:50s} | kept {len(coords):5d} / found {n_found:5d}")
        except Exception as e:
            coord_failures.append({"file_name": r.file_name, "error": repr(e)})
            print(f"FAIL {r.file_name[:50]} :: {e}")
    coords_path.write_text(json.dumps(all_coords))
    trunc_path.write_text(json.dumps(all_trunc))
    if coord_failures:
        pd.DataFrame(coord_failures).to_csv(NB_OUTPUT / "patch_coord_failures.csv", index=False)

n_trunc = sum(1 for v in all_trunc.values() if v["truncated"])
print(f"\nSlides with coordinates: {len(all_coords)}")
print(f"Slides truncated at cap: {n_trunc} / {len(all_trunc)}")

TRUNC TCGA-38-4631-01Z-00-DX1.5e0c873a-9c4c-4e0b-bf2e-e3 | kept  2000 / found  2000
TRUNC TCGA-44-6147-01Z-00-DX1.a34a9f49-e86b-43ee-9e2d-7e | kept  2000 / found  2000
TRUNC TCGA-44-2659-01Z-00-DX1.b33f40e6-bc18-48b0-8dad-bd | kept  2000 / found  2000
TRUNC TCGA-05-4396-01Z-00-DX1.49DD5F68-7473-4945-B384-EA | kept  2000 / found  2000
TRUNC TCGA-38-4629-01Z-00-DX1.d00cc280-5370-4b9f-9655-6c | kept  2000 / found  2000
TRUNC TCGA-44-8119-01Z-00-DX1.1EBEBFA7-22DB-4365-9DF8-C4 | kept  2000 / found  2000
TRUNC TCGA-05-4434-01Z-00-DX1.f3f5c89c-306f-4876-98fa-f7 | kept  2000 / found  2000
TRUNC TCGA-05-4402-01Z-00-DX1.c653ddc2-88c1-45ac-88e7-4e | kept  2000 / found  2000
TRUNC TCGA-44-5643-01Z-00-DX1.83fb7631-0ecb-4473-bfb2-c8 | kept  2000 / found  2000
TRUNC TCGA-44-2655-01Z-00-DX1.ee255271-780c-461c-ab23-5c | kept  2000 / found  2000
TRUNC TCGA-05-4398-01Z-00-DX1.269bc75f-492e-48b1-87ee-85 | kept  2000 / found  2000
TRUNC TCGA-05-4244-01Z-00-DX1.d4ff32cd-38cf-40ea-8213-45 | kept  2000 / foun

## PROCESSING — patch-level QC table

In [ ]:
# ============================================================
# PROCESSING — patch-level QC table (with progress bars)
# ============================================================
from time import time

try:
    from tqdm.auto import tqdm
    _HAS_TQDM = True
except Exception:
    _HAS_TQDM = False
    def tqdm(x, **kwargs):  # minimal fallback so the cell always runs
        return x

patch_manifest_path = NB_OUTPUT / "patch_manifest.csv"
patch_quality_path  = NB_OUTPUT / "patch_quality.csv"

if patch_manifest_path.exists() and patch_quality_path.exists():
    patch_df   = pd.read_csv(patch_manifest_path)
    quality_df = pd.read_csv(patch_quality_path)
    print(f"Loaded cached patch manifest: {len(patch_df)} rows")
else:
    manifest_rows = []
    quality_rows  = []

    # Outer progress bar: one tick per slide.
    slide_iter = tqdm(
        list(manifest.itertuples()),
        desc="Slides",
        unit="slide",
        dynamic_ncols=True,
        disable=not _HAS_TQDM,
    )

    t_start = time()
    for r in slide_iter:
        coords = all_coords.get(r.file_name, [])
        if not coords:
            print(f"WARNING: no coordinates for {r.file_name}")
            continue

        trunc_info = all_trunc.get(r.file_name, {})

        # Nested progress bar: one tick per patch on this slide.
        patch_iter = tqdm(
            coords,
            desc=f"  {r.file_name[:26]}",
            unit="patch",
            leave=False,
            dynamic_ncols=True,
            disable=not _HAS_TQDM,
        )

        slide = openslide.OpenSlide(str(r.svs_path))
        try:
            for (x, y, tf) in patch_iter:
                region = slide.read_region((x, y), 0, (PATCH_SIZE, PATCH_SIZE)).convert("L")
                arr = np.array(region)
                manifest_rows.append({
                    "file_name":             r.file_name,
                    "patient_id":            r.patient_id,
                    "case_number":           int(r.case_number),
                    "x":                     int(x),
                    "y":                     int(y),
                    "tissue_fraction":       float(tf),
                    "slide_truncated":       bool(trunc_info.get("truncated", False)),
                    "already_in_dev_cohort": bool(getattr(r, "already_in_dev_cohort", False)),
                })
                quality_rows.append({
                    "file_name":      r.file_name,
                    "x":              int(x),
                    "y":              int(y),
                    "mean_intensity": float(arr.mean()),
                    "std_intensity":  float(arr.std()),
                })
        finally:
            slide.close()

        # Update the outer bar's postfix with running totals.
        if _HAS_TQDM:
            elapsed = time() - t_start
            slide_iter.set_postfix({
                "patches": len(manifest_rows),
                "elapsed_s": f"{elapsed:.0f}",
            })

    patch_df   = pd.DataFrame(manifest_rows)
    quality_df = pd.DataFrame(quality_rows)

    patch_df.to_csv(patch_manifest_path, index=False)
    quality_df.to_csv(patch_quality_path, index=False)

    print(f"Saved: {patch_manifest_path} ({len(patch_df)} rows)")
    print(f"Saved: {patch_quality_path} ({len(quality_df)} rows)")

# Convenience merged view for the QC plots in this notebook (kept in-memory only)
patch_view = patch_df.merge(quality_df, on=["file_name", "x", "y"], how="left")
display(patch_view.head())

Slides:   0%|          | 0/98 [00:00<?, ?slide/s]

  TCGA-38-4631-01Z-00-DX1.5e:   0%|          | 0/2000 [00:00<?, ?patch/s]

  TCGA-44-6147-01Z-00-DX1.a3:   0%|          | 0/2000 [00:00<?, ?patch/s]

  TCGA-44-2659-01Z-00-DX1.b3:   0%|          | 0/2000 [00:00<?, ?patch/s]

  TCGA-05-4396-01Z-00-DX1.49:   0%|          | 0/2000 [00:00<?, ?patch/s]

  TCGA-38-4629-01Z-00-DX1.d0:   0%|          | 0/2000 [00:00<?, ?patch/s]

... (28 outputs left out)

  TCGA-05-4433-01Z-00-DX1.58:   0%|          | 0/2000 [00:00<?, ?patch/s]

  TCGA-38-4625-01Z-00-DX1.ff:   0%|          | 0/2000 [00:00<?, ?patch/s]

  TCGA-38-4626-01Z-00-DX1.14:   0%|          | 0/2000 [00:00<?, ?patch/s]

  TCGA-05-4426-01Z-00-DX1.4c:   0%|          | 0/2000 [00:00<?, ?patch/s]

## PROCESSING — foundation-model embeddings (OPTIONAL, off by default)

In [ ]:
# ============================================================
# PROCESSING — foundation-model embeddings (OPTIONAL)
# ============================================================
embed_dir = NB_OUTPUT / "patch_embeddings"
embed_dir.mkdir(parents=True, exist_ok=True)

if USE_FOUNDATION_MODEL:
    import torch
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device} | Model: {FOUNDATION_MODEL_NAME}")
    # Explicit stub: weights must be supplied locally.
    for r in manifest.itertuples():
        out_file = embed_dir / f"{r.file_name}.npy"
        if out_file.exists():
            print(f"cached:  {r.file_name[:60]}")
            continue
        print(f"pending (weights required): {r.file_name[:60]}")
    (embed_dir / "_ENABLED").write_text("Foundation model run was enabled.\n")
else:
    print("Foundation model disabled (USE_FOUNDATION_MODEL=False).")
    print("Patch coordinates and QC tables are still produced.")
    print("Embeddings will be generated later once weights are cached.")
    (embed_dir / "_DISABLED").write_text(
        "Foundation model was disabled for this run. Downstream notebooks "
        "should treat patch_embeddings/ as unavailable.\n"
    )

In [ ]:
# ============================================================
# EXPORT & DOWNLOAD ARTIFACT BUNDLE
# ============================================================
import shutil
from pathlib import Path
try:
    from IPython.display import FileLink, display
except ImportError:
    def FileLink(p): return p
    def display(*a): pass

export_dir = OUTPUT_ROOT / 'exports'
export_dir.mkdir(parents=True, exist_ok=True)
zip_name = f'verionco_outputs_{NB_OUTPUT.name}'
zip_path = export_dir / zip_name

# Package NB_OUTPUT into zip
shutil.make_archive(str(zip_path), 'zip', NB_OUTPUT)
final_zip = export_dir / f'{zip_name}.zip'

print('=' * 60)
print(f'VERIONCO NOTEBOOK {NB_OUTPUT.name} COMPLETE')
print(f'Artifact bundle created: {final_zip}')
print(f'Archive size: {final_zip.stat().st_size / (1024*1024):.2f} MB')
print('=' * 60)

# Display download link (clickable in Kaggle/Jupyter)
display(FileLink(str(final_zip)))
